<a href="https://colab.research.google.com/github/treeleaves30760/NTHU-NLP-HW2-Arithmetic-113061638/blob/main/NLP_HW2_NTHU_HW2_113061638_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive')

folder_path = os.path.join('gdrive', 'MyDrive', 'Graduate_School', 'NLP', 'HW2')
os.chdir(folder_path)
print("目前所在路徑:", os.getcwd())

In [ ]:
# Create a RNN to train Arithmetic

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
import numpy as np
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
def create_vocab() -> Tuple[Dict[str, int], Dict[int, str]]:
    chars = ['<pad>', '<eos>'] + [str(i) for i in range(10)] + list('+-*()=,.')
    char_to_id = {c: i for i, c in enumerate(chars)}
    id_to_char = {i: c for c, i in char_to_id.items()}
    return char_to_id, id_to_char

char_to_id, id_to_char = create_vocab()
print(f'char_to_id: {char_to_id}')
print(f'id_to_char: {id_to_char}')

In [ ]:
# Custom Dataset
class ArithmeticDataset(Dataset):
    def __init__(self, data: pd.DataFrame, char_to_id: Dict[str, int]):
        self.data = data
        self.char_to_id = char_to_id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if idx >= len(self.data):
            raise IndexError(f"Index {idx} is out of bounds for dataset with size {len(self.data)}")

        input_str, target_str = str(self.data.iloc[idx, self.data.columns.get_loc('src')]), str(self.data.iloc[idx, self.data.columns.get_loc('tgt')])

        input_seq = self.string_to_sequence(input_str)
        target_seq = self.string_to_sequence(target_str)

        input_seq = torch.tensor(input_seq, dtype=torch.long)
        target_seq = torch.tensor(target_seq, dtype=torch.long)

        input_seq = pad_sequence([input_seq], batch_first=True, padding_value=self.char_to_id['<pad>'])[0]
        target_seq = pad_sequence([target_seq], batch_first=True, padding_value=self.char_to_id['<pad>'])[0]

        return input_seq, target_seq

    def string_to_sequence(self, text: str) -> List[int]:
        sequence = [self.char_to_id[char] for char in text]
        sequence.append(self.char_to_id['<eos>'])
        return sequence


train_df = pd.read_csv('arithmetic_train.csv')
eval_df = pd.read_csv('arithmetic_eval.csv')

train_dataset = ArithmeticDataset(train_df, char_to_id)
eval_dataset = ArithmeticDataset(eval_df, char_to_id)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
eval_loader = DataLoader(eval_dataset, batch_size=128, num_workers=4)

In [ ]:
# LSTM Model
class ArithmeticLSTM(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor, hidden=None) -> Tuple[torch.Tensor, Tuple]:
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        output = self.fc(lstm_out)
        return output, hidden

model = ArithmeticLSTM(
        vocab_size=len(char_to_id),
        embedding_dim=256,
        hidden_dim=512,
    ).to(device)

In [ ]:
def train_epoch(model: nn.Module,
                dataloader: DataLoader,
                criterion: nn.Module,
                optimizer: torch.optim.Optimizer,
                device: torch.device) -> float:
    model.train()
    total_loss = 0

    for batch_idx, (input_seq, target_seq) in enumerate(dataloader):
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)

        optimizer.zero_grad()
        output, _ = model(input_seq)

        # Reshape output and target for loss calculation
        output = output.view(-1, output.shape[-1])
        target_seq = target_seq.view(-1)

        loss = criterion(output, target_seq)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')

    return total_loss / len(dataloader)

def evaluate(model: nn.Module,
      dataloader: DataLoader,
      id_to_char: Dict[int, str],
      device: torch.device) -> float:
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for input_seq, target_seq in dataloader:
            input_seq = input_seq.to(device)
            batch_size = input_seq.shape[0]

            # Generate sequence
            outputs = []
            hidden = None
            current_input = input_seq[:, 0].unsqueeze(1)

            for _ in range(input_seq.shape[1]):
                output, hidden = model(current_input, hidden)
                pred = output.argmax(dim=-1)
                outputs.append(pred)
                current_input = pred

            # Convert predictions to strings
            pred_seqs = torch.cat(outputs, dim=1).cpu()
            pred_strings = [''.join([id_to_char[id.item()] for id in seq]).rstrip('<pad>')
                          for seq in pred_seqs]

            # Convert targets to strings
            target_strings = [''.join([id_to_char[id.item()] for id in seq]).rstrip('<pad>')
                            for seq in target_seq]

            # Compare predictions with targets
            correct += sum(1 for p, t in zip(pred_strings, target_strings) if p == t)
            total += batch_size

    return correct / total

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore padding
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)

In [ ]:
# Training loop
num_epochs = 10
best_accuracy = 0

for epoch in tqdm(range(num_epochs)):

    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    print(f'Training Loss: {train_loss:.4f}')

    # Evaluate
    accuracy = evaluate(model, eval_loader, id_to_char, device)
    print(f'Evaluation Accuracy: {accuracy:.4f}')

    # Learning rate scheduling
    scheduler.step(train_loss)

    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': accuracy,
            'char_to_id': char_to_id,
            'id_to_char': id_to_char
        }, 'best_model.pt')

    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'accuracy': accuracy,
        'char_to_id': char_to_id,
        'id_to_char': id_to_char
    }, f'checkpoint_epoch_{epoch+1}.pt')